In [1]:
%pip install -qU langchain-google-genai langchain-pinecone langchain-community pypdf pinecone-client

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
spyder 6.0.7 requires bcrypt>=4.3.0, but you have bcrypt 4.0.1 which is incompatible.
streamlit 1.51.0 requires pyarrow<22,>=7.0, but you have pyarrow 25.0.1 which is incompatible.


In [ ]:
import api_key
from langchain_openai import ChatOpenAI
import os
index_name = "crime-manual"

In [4]:
# 导入必要的解析pdf文档库
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

C:\Users\yang sunyou\AppData\Local\Temp\ipykernel_51232\593056986.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
#解析传入的pdf文档，切块，使用embedding模型生成向量
print('解析传入的pdf文档…')
loader=PyPDFLoader("./Crime Classification Manual - A Standard System for Investigating and Classifying Violent Crime - 3E (2013).pdf")
docs=loader.load()
print('文档解析完成，共加载了', len(docs), '页文档。')



解析传入的pdf文档…
文档解析完成，共加载了 578 页文档。


In [10]:
#对文档切块
print('对文档进行切块处理…')
#chunk_size:每块保留500字符，
#chunk_overlap:前后两块重叠50个字符，防止关键句被切断
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks=text_splitter.split_documents(docs)
print(f'文档已经切分成{len(chunks)}个数据块')

对文档进行切块处理…
文档已经切分成3436个数据块


In [15]:
# 启动 Gemini embedding model 并将文档导入 Pinecone 向量数据库
print('正在连接 Gemini embedding model 并导入 Pinecone 向量数据库...')
embeddings = GoogleGenerativeAIEmbeddings(
    model='models/gemini-embedding-001',
    # output_dimensionality=768,
)
vector_store = PineconeVectorStore.from_documents(
    chunks, embeddings, index_name=index_name
)
print('已成功将文档导入 Pinecone 向量数据库。')

正在连接 Gemini embedding model 并导入 Pinecone 向量数据库...


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0\nPlease retry in 56.959935446s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerMinutePerUserPerProjectPerModel-FreeTier', 

In [16]:
# 由于速率限制，这里改用开源模型BAAI/bge-base-en-v1.5
%pip install -qU sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [17]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

print('使用本地开源模型 BAAI/bge-base-en-v1.5')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'当前 embedding 设备: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-base-en-v1.5',
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True},
)
vector_store = PineconeVectorStore.from_documents(
    chunks,
    embeddings,
    index_name=index_name,
)

print('已成功将文档导入 Pinecone 向量数据库。')

使用本地开源模型BAAI/bge-base-en-v1.5


W0927 06:20:21.158000 51232 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0927 06:20:21.270000 51232 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

d:\Anaconda\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI_Cache\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

... (7 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

已成功将文档导入 Pinecone 向量数据库。


In [31]:
#为数据库配置检索器，每次提问返回最相关的3个数据块
retriever= vector_store.as_retriever(
    search_kwargs={"k": 8,
                   'fetch':30
                   },
    search_type="mmr",
                                     )

In [32]:
# 单独测试检索阶段，查看召回的 3 个片段
test_query = "这份文档最核心的主旨是什么？请列出两个关键点。"
retrieved_docs = retriever.invoke(test_query)

print(f"共召回 {len(retrieved_docs)} 个文本块：\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- 召回片段 {i+1} ---")
    print(doc.page_content[:300])  # 打印前300个字符
    print("\n")

共召回 8 个文本块：

--- 召回片段 1 ---
Retri eved on August 2, 2012, fro m http://news.goog l e.com/n ewspapers?nid¼861 &dat ¼
2003 0828&i d¼CXpTAAA AIBAJ&sjid¼pYUD AAAAI BAJ&pg¼6883,7 169874
Wilder, S. H. (1981). International terrorism and hostage-taking: An overview.Manitoba Law Journal,
4(1), 367 –376.
Zhang, S., & Chin, K. (2002). E


--- 召回片段 2 ---
Permissions Department, John Wiley & Sons, Inc., 111 River Street, Hoboken, NJ 07030,
(201) 748-6011, fax (201) 748-6008, or online at www.wiley.com/go/permissions.
Limit of Liability/Disclaimer of Warranty: While the publisher and author have used their best
efforts in preparing this book, they mak


--- 召回片段 3 ---
3GC13 03/02/2013 11:43:20 Page 375
week to ﬁnd a broken pencil placed visibly on his appointment book. In another
case, a patient sent his therapist of two years ﬂowers, gifts, and cards. He
called her on several occasions and left messages using a pseudonym on her
answering machine. When she did no


--- 召回片段 4 ---
the text.
277


--

In [ ]:
# 初始化Gemini flash 模型 temp=0 保证回答基于客观事实，不发散
# llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

# 替换成deep seek模型
llm = ChatOpenAI(
    model_name="deepseek-flash",
    openai_api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    max_tokens=1024,
)


# 定义RAG提示词模板

template = """你是一个专业文档问答助手，基于以下给出的背景信息回答用户的问题。
要求：
1. 使用中文回答；
2. 只依据资料内容，不要编造；
3. 如果资料中没有答案，请明确说“资料中没有找到相关信息”；
4. 必要时保留英文专业术语，并在后面给出中文解释。

背景信息:{context}

用户问题:{question}

最终回答：
"""
prompts=PromptTemplate.from_template(template)

# 使用LCEL(Long chain expression language)语法拼接完整的RAG工作流
rag_chain=(
    {'context':retriever,'question':RunnablePassthrough()}
    |prompts
    |llm
    |StrOutputParser()
)

print("\n--- 系统准备就绪，开始执行问答测试 ---")


--- 系统准备就绪，开始执行问答测试 ---


In [35]:
# 1. 模拟中间件的翻译过程
test_question = (
    "How is the crime scene behavior of serial killers classified in the manual?"
)
print(f"测试用的英文查询词：{test_question}\n")

# 2. 模拟检索，看看 Pinecone 到底吐出了哪 3 段原文
retrieved_docs = retriever.invoke(test_question)
print(
    "--- 以下是 Pinecone 实际从《Crime Classification Manual》中召回的英文原文片段 ---"
)
for i, doc in enumerate(retrieved_docs):
    print(f"\n[片段 {i+1}]")
    print(doc.page_content)  # 打印完整的被切分的原文

测试用的英文查询词：How is the crime scene behavior of serial killers classified in the manual?

--- 以下是 Pinecone 实际从《Crime Classification Manual》中召回的英文原文片段 ---

[片段 1]
Organization of the Manual
This third edition of the Crime Classi ﬁcation Manual is divided into three
major sections. The ﬁrst section of the manual contains ﬁve chapters on Crime
Analysis and Investigation. Part II includes the classiﬁ cation categories of
Homicide, Arson/Bombing, Rape and Sexual Assault, Nonlethal Crimes,
Computer Crimes, Global Crimes, Mass and Serial Homicide and Poisoning
and Biological Agents as Weapons. Part III includes Legal Issues of Interview-

[片段 2]
some clearly identi ﬁable form of psychopathology. In such a case, the crime
scene is presumed to re ﬂect the murderer ’s behavior and personality in much
the same way as furnishings reveal a homeowner ’s character.
Proﬁling is, in fact, a form of retroclassi ﬁcation, or classiﬁcation that works
backward. Typically, we classify a known entity into a disc

In [34]:

user_query = "How is the crime scene behavior of serial killers classified in the manual?"
response = rag_chain.invoke(user_query)

print(f"用户提问: {user_query}")
print(f"AI 回答:\n{response}")

用户提问: How is the crime scene behavior of serial killers classified in the manual?
AI 回答:
资料中没有找到关于手册中连环杀手犯罪现场行为具体如何分类的相关信息。


In [29]:
# 目的是为了解决中文提问导致的检索不准确问题，
# 其逻辑是：用户输入中文-> Gemini 毫秒级翻译为精准的英文检索词 -> 送入 Pinecone 检索 -> 拿到英文原文 -> Gemini 用中文总结输出。
from langchain_core.runnables import RunnableLambda


translator_prompt = PromptTemplate.from_template(
    "You are a translation assistant. Translate the following user query into a natural, grammatically correct English question. DO NOT output a list of keywords. Only output the translated English sentence, nothing else.\nQuery: {question}"
)

query_translator=translator_prompt|llm|StrOutputParser()


# 检索前先翻译
def retrieve_with_translation(question: str):
    # 先把中文转换成英文
    en_query=query_translator.invoke({"question": question}).strip()
    print(f"[系统中间件] 原始提问: {question}")
    print(f"[系统中间件] 转换后的英文检索词: {en_query}")

    # 使用英文检索词去 Pinecone 查英文文档
    return retriever.invoke(en_query)

# 组装RAG链
enhanced_rag_chain=(
    {'context':RunnableLambda(retrieve_with_translation),'question': RunnablePassthrough()}
    |prompts
    |llm
    |StrOutputParser()
)

In [ ]:
user_query = "手册中对连环杀手的现场行为是如何分类的？"
response = enhanced_rag_chain.invoke(user_query)

print(f"\nAI 最终回答:\n{response}")

[系统中间件] 原始提问: 手册中对连环杀手的现场行为是如何分类的？
[系统中间件] 转换后的英文检索词: How is the crime scene behavior of serial killers classified in the manual?

AI 最终回答:
资料中没有找到相关信息。提供的资料中仅提及了早期针对系列性谋杀分类系统的开发历史，以及“有组织型犯罪者”（organized offender，指有计划、有预谋且在现场表现出控制力的作案者）的部分特征，但并未包含手册对连环杀手现场行为具体分类方式的完整内容。


In [39]:
# 由于pypdf的暴力分割导致内容不完整，以至于检索结果不够准确，这里更高级的文档结构化解析工具来重构这部分代码
%pip install -qU langchain-unstructured "unstructured[pdf]"
%pip install  tesseract

Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://pypi.org/simple/
     ---------------------------------------- 0.0/45.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/45.6 MB ? eta -:--:--
     ---------------------------------------- 0.0/45.6 MB ? eta -:--:--
     ---------------------------------------- 0.3/45.6 MB ? eta -:--:--
     ---------------------------------------- 0.5/45.6 MB 1.9 MB/s eta 0:00:24
      --------------------------------------- 1.0/45.6 MB 2.1 MB/s eta 0:00:22
     - -------------------------------------- 1.8/45.6 MB 2.6 MB/s eta 0:00:17
     - -------------------------------------- 2.1/45.6 MB 2.5 MB/s eta 0:00:18
     -- ------------------------------------- 3.1/45.6 MB 2.9 MB/s eta 0:00:15
     --- ------------------------------------ 3.9/45.6 MB 3.1 MB/s eta 0:00:14
     ---- ----------------------------------- 4.7/45.6 MB 3.3 MB/s eta 0:00:13
     ----- ---------------------------------- 5.8/45.6 MB 3.4 MB/s eta 0:00:12
     ----- ------------------------

  DEPRECATION: Building 'tesseract' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'tesseract'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [41]:
%pip install -qU "unstructured[pdf]" unstructured-inference

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.8.0 which is incompatible.


In [42]:
from langchain_unstructured import UnstructuredLoader

print("正在使用 Unstructured 进行语义感知解析与切块...")
loader = UnstructuredLoader(
    file_path="./Crime Classification Manual - A Standard System for Investigating and Classifying Violent Crime - 3E (2013).pdf",
    strategy="fast",
    chunking_strategy="by_title",
    # 物理限制兜底：就算一个章节很长，每个块最多也不超过 800 字符
    max_characters=800,
    # 语义拼图：如果某个标题下只有两句话（比如低于 200 字符），自动把它和相邻短段落合并，防止碎片化
    combine_text_under_n_chars=200,
)

# 这一步会一次性完成解析 + 结构识别 + 语义切块
semantic_chunks = loader.load()


print(
    f"解析完成！500页的 PDF 被智能重组为 {len(semantic_chunks)} 个带语义上下文的数据块。"
)

# 随便打印一个切块，看看它的威力
print("\n--- 抽查一个语义切块 ---")
print("切块内容:\n", semantic_chunks[100].page_content)
print("\n附带元数据:\n", semantic_chunks[100].metadata)

正在使用 Unstructured 进行语义感知解析与切块...


INFO: generated new fontManager
INFO: pikepdf C++ to Python logger bridge initialized
INFO: Downloading spaCy model en_core_web_sm 3.8.0 …
INFO: Installed en_core_web_sm 3.8.0


解析完成！500页的 PDF 被智能重组为 3065 个带语义上下文的数据块。

--- 抽查一个语义切块 ---
切块内容:
 The investigator should remember that the apparent absence of penetration with the penis does not mean the victim was not sexually assaulted. Sexual assault also includes insertion of foreign objects, regressive necrophilia, and many activities that target the breasts, buttocks, and genitals.

附带元数据:
 {'source': './Crime Classification Manual - A Standard System for Investigating and Classifying Violent Crime - 3E (2013).pdf', 'file_directory': '.', 'filename': 'Crime Classification Manual - A Standard System for Investigating and Classifying Violent Crime - 3E (2013).pdf', 'last_modified': '2026-09-24T04:26:11', 'page_number': 27, 'orig_elements': 'eJxtUk2PmzAQ/SsjTq0UEDhsssmtWvXQQ6tKiXqJVtGAJ+AWbGQPSdBq/3vHsOmpF+R58/HmveH0llBHPVk+G53sIcEtYr5T2xSLEtOy2FYpbnZVeil3FWmsS7VRyQqSnhg1MkrPW1I757WxyBTmuMPJjXxuyTQtC7JVKnteP0nbR+ZmNLeSeMo3H4nBGcux+3TaFVm5AqV2mXpdwSPcPmd5DMu1yor/xHO5AEmYAlMftfw0d+oOA9aUvEviYjo6a+OpZuenWJAlH7DFniLw4

In [43]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

print("使用本地开源模型 BAAI/bge-base-en-v1.5")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"当前 embedding 设备: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)
vector_store = PineconeVectorStore.from_documents(
    semantic_chunks,
    embeddings,
    index_name="crime-manual-semantic",
)

print("已成功将文档导入 Pinecone 向量数据库。")

使用本地开源模型 BAAI/bge-base-en-v1.5
当前 embedding 设备: cpu


INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/config_se

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO: HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD h

已成功将文档导入 Pinecone 向量数据库。


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("1. 重新绑定检索器到全新的语义向量库...")
retriever = vector_store.as_retriever(
    search_type="mmr",# 使用最大边际相关性（MMR）算法进行检索
    search_kwargs={"k": 8, "fetch_k": 30}
)

print("2. 重新装配翻译中间件与检索函数...")


def retrieve_with_translation(query: str):
    # 此处的 query_translator 是之前已经定义好的
    english_query = query_translator.invoke({"question": query}).strip()
    print(f"[系统中间件] 原始提问: {query}")
    print(f"[系统中间件] 转换后的英文检索词: {english_query}")
    # 此处的 retriever 已经是全新绑定的版本
    return retriever.invoke(english_query)


print("3. 重新组装最终的 RAG 问答链...")
enhanced_rag_chain = (
    {
        "context": RunnableLambda(retrieve_with_translation),
        "question": RunnablePassthrough(),
    }
    | prompts
    | llm
    | StrOutputParser()
)


INFO: AFC is enabled with max remote calls: 10.


1. 重新绑定检索器到全新的语义向量库...
2. 重新装配翻译中间件与检索函数...
3. 重新组装最终的 RAG 问答链...

--- 系统重组完毕，开始极限测试 ---


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 200 OK"


[系统中间件] 原始提问: 手册中对连环杀手的现场行为是如何分类的？
[系统中间件] 转换后的英文检索词: How does the manual classify the crime scene behavior of serial killers?


INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 200 OK"



AI 最终回答:
资料中没有找到相关信息。


In [46]:
# 1. 打印刚才失败的查询，看看 Unstructured 切出的新块到底长什么样
failed_query = (
    "How does the manual classify the crime scene behavior of serial killers?"
)
print("--- 失败查询的实际召回片段（查看语义切块效果） ---")
docs = retriever.invoke(failed_query)
for i, doc in enumerate(docs[:3]):  # 只看前 3 个
    # 注意这里：你会看到 Unstructured 成功保留了 metadata 中的各种结构信息
    print(f"\n[片段 {i+1}]")
    print(doc.page_content[:400] + "...")


# 2. 修正提问策略：直接使用符合 FBI 体系的专业本体术语
# 我们不再强行绑定“serial killers”，而是直接问“犯罪现场的组织和无组织状态”
accurate_query = "手册中关于犯罪现场（crime scene）的组织型（organized）、无组织型（disorganized）和混合型（mixed）分类，各自有哪些特征？"

print("\n\n======================================================\n")
print(f"正在使用对齐领域本体的新问题发起测试：\n{accurate_query}")

# 调用我们组装好的增强型 RAG 链
final_response = enhanced_rag_chain.invoke(accurate_query)
print(f"\nAI 最终回答:\n{final_response}")

--- 失败查询的实际召回片段（查看语义切块效果） ---


INFO: AFC is enabled with max remote calls: 10.



[片段 1]
In the development of the manual, a decision was made to base the classiﬁcation on the primary intent of the criminal. The intent categories include: (1) criminal enterprise, (2) personal cause, (3) sexual intent, and (4) group cause. Task force groups chaired by supervisory special agents at the FBI’s National Center for the Analysis of Violent crime worked on reﬁning the crime categories for the...

[片段 2]
A major part of the process of crime scene analysis depends on the analyst’s insight into the dynamics of human behavior. Speech patterns, writing styles, verbal and nonverbal gestures, and other traits and patterns compose human behavior. This combination causes every individual to act, react, function, or perform in a unique and speciﬁc way. This individualistic behavior usually remains consiste...

[片段 3]
454 CRIME CLASSIFICATION MANUAL

CRIME SCENE INDICATORS FREQUENTLY NOTED...



正在使用对齐领域本体的新问题发起测试：
手册中关于犯罪现场（crime scene）的组织型（organized）、无组织型（disorganized）和混合型（mixed）分类

INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 200 OK"


[系统中间件] 原始提问: 手册中关于犯罪现场（crime scene）的组织型（organized）、无组织型（disorganized）和混合型（mixed）分类，各自有哪些特征？
[系统中间件] 转换后的英文检索词: According to the manual, what are the respective characteristics of organized, disorganized, and mixed crime scenes?


INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 200 OK"



AI 最终回答:
基于所提供的资料，关于犯罪现场（crime scene）在组织型、无组织型和混合型分类下的特征如下：

需要说明的是，手册指出犯罪现场极少是完全组织型或完全无组织型的，通常处于两者极端之间的连续体（continuum）上，其位置取决于罪犯的预谋程度、策划程度及其能力。

### 1. 组织型犯罪现场（Organized Crime Scene）
* **现场状态与控制：** 现场通常有序且整洁（orderly, neat）；反映出罪犯具有较高的犯罪老练程度（criminal sophistication）、对受害人具备良好的控制力，以及案件中涉及较多的预谋（premeditation）和策划。
* **物证与手法（以纵火等类型为例）：** 现场遗留的物证较少；如果存在强行闯入，手法会更为娴熟（more skillful）；使用精心制作的纵火/引火装置（elaborate incendiary devices，例如电子定时装置、引火器等）。
* **特殊情况（如准军事极端组织犯罪）：** 犯罪现场通常高度组织化，展现出军事战术和作案手法（modus operandi, MO，通常简称为 MO），且不会进行伪装/布置现场（staging），因为其杀戮目的是传达某种信息。

### 2. 无组织型犯罪现场（Disorganized Crime Scene）
* **现场状态：** 现场极度混乱、杂乱无章（disarrayed, sloppy / show great disarray）。
* **尸体处置：** 尸体通常没有被藏匿或掩盖（not concealed），而是被遗弃在开阔处或极容易被发现的地方。
* **凶器使用：** 凶器通常是就地取材的应急武器（weapon of opportunity，例如石头），且会直接遗留在犯罪现场；经常使用人体自身部位作为武器（personal weapon use，如拳头和脚）。
* **法医物证发现：** 法医鉴定通常会显示出突袭式攻击（blitz style of attack）、过度杀戮（overkill，指超出致死必要的暴力）以及钝器创伤（blunt-force trauma）。

### 3. 混合型犯罪现场（Mixed Crime Scene）
* **行为表现不一致：** 罪犯在作案过程中表现出

In [47]:


print("\n--- 系统重组完毕，开始极限测试 ---")
user_query = "手册中对杀手的现场行为是如何分类的？"
response = enhanced_rag_chain.invoke(user_query)

print(f"\nAI 最终回答:\n{response}")

INFO: AFC is enabled with max remote calls: 10.



--- 系统重组完毕，开始极限测试 ---


INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 200 OK"


[系统中间件] 原始提问: 手册中对杀手的现场行为是如何分类的？
[系统中间件] 转换后的英文检索词: How is a killer's crime scene behavior classified in the manual?


INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent "HTTP/1.1 429 Too Many Requests"
INFO: Retrying google.genai._api_client.BaseApiClient._request_once in 1.859143377411417 seconds as it raised ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 59.774440302s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-lim

KeyboardInterrupt: 

In [48]:
%pip install -qU langchain-openai

Note: you may need to restart the kernel to use updated packages.
